In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/quikr_car.csv')
df.head(3)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['year'].unique()

In [ ]:
df['Price'].unique()

In [ ]:
df['kms_driven'].unique()

In [ ]:
df['fuel_type'].unique()

array(['Petrol', 'Diesel', nan, 'LPG'], dtype=object)

In [ ]:
# Quality
# year has many non-year values
# object to integer
# Price has ask for price
# object to integer and comma
# Kms has kms with integers
# Kms has nan values
# fuel_type has nan values
# Keep first 3 words of name

# Quality
###### year has many non-year values
###### object to integer
###### Price has ask for price
###### object to integer and comma
###### Kms has kms with integers
###### Kms has nan values
###### fuel_type has nan values
###### Keep first 3 words of name

# **Cleaning**

In [ ]:
backup=df.copy()

In [ ]:
df = backup.copy()
df=df[df['year'].str.isnumeric()]

In [ ]:
df['year']=df['year'].astype(int)

In [ ]:
df=df[df['Price']!='Ask For Price']

In [ ]:
df['Price']=df['Price'].str.replace(',','').astype(int)

In [ ]:
df['kms_driven']=df['kms_driven'].str.split(' ').str.get(0).str.replace(',','')

In [ ]:
df=df[df['kms_driven'].str.isnumeric()]

In [ ]:
df['kms_driven']=df['kms_driven'].astype(int)

In [ ]:
df=df[~df['fuel_type'].isna()]

In [ ]:
df['name']=df['name'].str.split(' ').str.slice(0,3).str.join(' ')

In [ ]:
df=df.reset_index(drop=True)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# df=df[df['Price']>6e6].reset_index(drop=True)

In [ ]:
df.to_csv('Cleaned_Car_data.csv')

# **Model**

In [ ]:
x=df.drop(columns='Price')
y=df['Price']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=661)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
categorical_features = ['name', 'company', 'fuel_type']
one_hot_encoder = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(transformers=[('cat', one_hot_encoder, categorical_features)],remainder='passthrough')

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
model.fit(x_train, y_train)
display('Model training complete!')

'Model training complete!'

In [ ]:
import joblib
print("joblib imported successfully.")

joblib imported successfully.


In [ ]:
joblib.dump(model, 'linear_regression_model.pkl')
print("Model saved successfully to linear_regression_model.pkl")

Model saved successfully to linear_regression_model.pkl


In [60]:
import streamlit as st
import joblib
import pandas as pd
import numpy as np

# This content would typically be saved to 'app.py' in a real deployment scenario.
# For the purpose of this interactive environment, I will output the script content.
# In a real scenario, you would create an 'app.py' file and put this code inside.

print("Code for app.py (imports):\n")
print("""import streamlit as st
import joblib
import pandas as pd
import numpy as np
""")

Code for app.py (imports):

import streamlit as st
import joblib
import pandas as pd
import numpy as np



In [ ]:
import sys
!{sys.executable} -m pip install streamlit
print("streamlit installed successfully.")

In [ ]:
app_py_content = """import streamlit as st
import joblib
import pandas as pd
import numpy as np

# Load the pre-trained model
model = joblib.load('linear_regression_model.pkl')

# Load the cleaned data to get unique values for dropdowns and ranges
df = pd.read_csv('Cleaned_Car_data.csv')

# Streamlit app title
st.title('Car Price Predictor')

# Input widgets
company = st.selectbox('Company', sorted(df['company'].unique()))
name = st.selectbox('Model', sorted(df[df['company'] == company]['name'].unique()))
year = st.number_input('Year', min_value=int(df['year'].min()), max_value=int(df['year'].max()), value=2015, step=1)
kms_driven = st.number_input('Kilometers Driven', min_value=0, value=50000, step=1000)
fuel_type = st.selectbox('Fuel Type', sorted(df['fuel_type'].unique()))

# Prediction button
if st.button('Predict Price'):
    # Create a DataFrame from user inputs
    input_data = pd.DataFrame([[name, company, year, kms_driven, fuel_type]],
                                columns=['name', 'company', 'year', 'kms_driven', 'fuel_type'])

    # Make prediction
    prediction = model.predict(input_data)[0]

    # Display the predicted price
    st.success(f'Predicted Price: ₹ {int(prediction):,}')

"""

# Print the content that would go into app.py
print("Code for app.py:\n")
print(app_py_content)


In [56]:
with open('app.py', 'w') as f:
    f.write(app_py_content)
print('app.py created successfully.')

app.py created successfully.


In [57]:
import sys
!{sys.executable} -m pip install pyngrok
print("pyngrok installed successfully.")

pyngrok installed successfully.


In [58]:
import subprocess

# Run the Streamlit app in the background
subprocess.Popen(['streamlit', 'run', 'app.py'])
print('Streamlit app started in the background.')

Streamlit app started in the background.


In [61]:
from pyngrok import ngrok

ngrok.set_auth_token('36VWgqTHaIoudSns3W1Llqe7NL4_82AnxyCspdRGoWFhrjjDZ')

# Connect to ngrok and expose the Streamlit app's default port (8501)
public_url = ngrok.connect('8501')
print(f"Streamlit App Public URL: {public_url}")

Streamlit App Public URL: NgrokTunnel: "https://overtimorous-stirringly-eleni.ngrok-free.dev" -> "http://localhost:8501"
